# P1 · Plan B probe: generated images as prototypes for label names (checklist 1.6)

No map fitted on ImageNet classes can know what an unseen name looks like, but a perfect map would still help
Seam 2 (NINCO FPR95 36.6 → 30.3 in the stream simulation). This probe asks whether a text-to-image model can
supply that knowledge: it draws a few images for each name with **SDXL-Turbo**, encodes them with our CLIP
ViT-B/16, and saves the embeddings. experiments/08_seam1_generation.py then compares these generated prototypes
with the real image centroids and with the perfect-map bound.

Names: all 64 NINCO classes, 300 random SSB-hard classes, and 50 random ImageNet classes (a sanity check:
CLIP should recognise generated ImageNet images). 4 images per name, 1,656 images in total.

**Runtime:** GPU. A100 or L4 if Colab offers one (5–15 minutes), otherwise T4 (about 30–40 minutes). The run
saves as it goes, so a dropped session resumes where it stopped. **Output:**
`MyDrive/ReNeg/transfers/analysis_pack2/reneg_gen_probe.npz` (about 3 MB), the input of the local evaluation.

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("P1_generation_probe", need_gpu=False)
log = ctx.log
import json, subprocess, time
import numpy as np, torch
if not torch.cuda.is_available():
    print("No GPU: Runtime -> Change runtime type -> A100, L4 or T4 (generation on a CPU takes hours)")
from oodlab.report import find_cache
from oodlab.clipwrap import load_clip
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation on small GPUs
from reneg.gen import clip_image_encoder, fake_generator, generate_safely, load_pipeline, probe_names, run_probe
from reneg.transport import l2n
from reneg.wordnet import get_wordnet
FAKE = os.environ.get("RENEG_FAKE_GEN") == "1"          # only for offline tests of this notebook
BACKBONE = "ViT-B/16"
OUT = f"{DRIVE_ROOT}/transfers/analysis_pack2"
os.makedirs(OUT, exist_ok=True)
cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
wn = get_wordnet()
print("device:", ctx.device, "| WordNet:", "ready" if wn is not None else "unavailable (SSB-hard names skipped)")

### Settings

`MODEL = "stabilityai/sd-turbo"` is about three times faster than SDXL-Turbo, if the T4 is too slow.

In [ ]:
MODEL = "stabilityai/sdxl-turbo"
N_PER = 4                 # images per name
SSB_MAX = 300             # random SSB-hard classes (all 64 NINCO classes are always included)
N_IMAGENET = 50           # ImageNet classes for the sanity check
PROMPT = "a photo of a {}."
BATCH = 4                 # halves itself automatically if the GPU runs out of memory
names, tags, folders = probe_names(cache, bank, wn=wn, ssb_max=SSB_MAX, n_imagenet=N_IMAGENET)
print({t: tags.count(t) for t in dict.fromkeys(tags)}, "names;", N_PER * len(names), "images to draw")
print("e.g.", [n for n, t in zip(names, tags) if t == "ninco"][:4], [n for n, t in zip(names, tags) if t == "ssb_hard"][:4])

### Load the generator and our CLIP image encoder

The generator downloads once per session (about 7 GB for SDXL-Turbo, 1–2 minutes). If the download asks for a
token, add a Hugging Face token as the Colab secret `HF_TOKEN` (key icon, Notebook access on) and run this cell
again. If an earlier run stopped with `CUDA out of memory`, first use **Runtime → Restart session**.

In [ ]:
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
except Exception:
    pass
WEIGHTS = f"{DRIVE_ROOT}/weights"
if FAKE:
    gen = fake_generator(64)
    dim = int(bank.id_text.shape[1])
    proj = torch.randn(3, dim, generator=torch.Generator().manual_seed(0))
    enc = lambda ims: l2n(torch.tensor(np.stack([np.asarray(im, np.float32).mean((0, 1)) for im in ims])) @ proj)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.27", "accelerate"], check=False)
    gen = load_pipeline(MODEL, device=ctx.device, log=log.info)
    model = load_clip(BACKBONE, device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS)
    enc = clip_image_encoder(model, device=ctx.device)
# quick check before the long run: CLIP should recognise generated ImageNet classes
probe_ids = [i for i, t in enumerate(tags) if t == "imagenet"][:8]
t0 = time.time()
ims = generate_safely(gen, [PROMPT.format(names[i]) for i in probe_ids], seed=123, log=print)
e = enc(ims)
pred = (e @ l2n(bank.id_text.float()).T).argmax(1).numpy()
ok = [int(folders[i]) == int(p) for i, p in zip(probe_ids, pred)]
print(f"{sum(ok)}/{len(ok)} generated ImageNet images recognised as their class; {time.time() - t0:.1f}s for 8 images")
if not FAKE and sum(ok) < 4:
    print("WARNING: CLIP recognises few generated images; check the model and the image preprocessing before the long run")

### Draw and encode (resumable)

In [ ]:
out_path = os.path.join(OUT, "reneg_gen_probe.npz")
info = run_probe(gen, enc, names, tags, folders, out_path, n_per=N_PER, batch=BATCH, prompt=PROMPT,
                 model_id=MODEL, log=log.info)
print(json.dumps(info, indent=1))

### Summary

In [ ]:
z = np.load(out_path)
E = torch.tensor(z["emb"]).float()
tg = np.asarray(z["tags"])[z["name_idx"]]
im = tg == "imagenet"
if im.any():
    truth = np.asarray([int(f) for f in np.asarray(z["folders"])[z["name_idx"][im]]])
    acc = 100 * float(((E[torch.as_tensor(im)] @ l2n(bank.id_text.float()).T).argmax(1).numpy() == truth).mean())
    print(f"generated ImageNet images recognised by zero-shot CLIP: {acc:.1f}%")
for t in ("ninco", "ssb_hard", "imagenet"):
    print(f"{t:9s} {int((tg == t).sum()):5d} images")
print(f"{out_path}: {os.path.getsize(out_path) / 1e6:.1f} MB (input of experiments/08_seam1_generation.py)")
finish(ctx, {"images": int(len(tg)), "model": MODEL})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")